# B2-024-gpu-scientific-ml-capstone — Practice p27 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** mixture-parameter-regression

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

import time

MAX_STEPS = 2000
MAX_SECONDS = 15.0

## Solution

**Idea: peak-finding initialization + batched least-squares refinement.** The fixed-initialization baseline fails when its starting centers $0.25/0.5/0.75$ are in the wrong basin: a component gets stuck between two bumps or two components chase one bump. But the generator guarantees well-separated bumps (center gaps $\ge3\times$ the largest width, i.e. at least about 4 sample spacings), and the noise ($0.02$) is small next to the weights ($\ge0.5$), so the three highest **local maxima** of the 32 samples sit almost on top of the three true centers. Starting each component at a local maximum (weight = sampled height, width = the median training width) puts every fit in the right basin; a short refinement then fits each function to its 32 samples.

`fit` learns the only quantities the initializer needs from the training rows: the median component width (starting width) and the minimum weight (a floor for the starting height). Refinement uses `capstone_data.mixture_ls_fit` with the budget's `optimizer_wrapper`, one step per batched update.

**Budget plan** (one `StepBudget(2000)` created first): main approach 150 refinement steps on validation and 150 inside the test call; alternative 1 (fixed initialization, same 150 steps) 150 steps; alternative 2 (amortized MLP regressor) 600 training steps + 150 refinement steps. Total 1,200 steps.

In [ ]:
budget = capstone_data.StepBudget(MAX_STEPS)
train_x, train_y = capstone_data.train_rows("mixture")
val_x, val_y = capstone_data.val_rows("mixture")
X = capstone_data.MIXTURE_X
SEAM = []
REFINE_STEPS, REFINE_LR = 150, 0.02


class PeakInitLeastSquares:
    def __init__(self, budget, refine_steps=REFINE_STEPS, refine_lr=REFINE_LR, radius=2):
        self.budget, self.refine_steps, self.refine_lr, self.radius = budget, refine_steps, refine_lr, radius

    def fit(self, train_x, train_y):
        SEAM.extend(capstone_data.split_of("mixture", row) for row in train_x)
        p = train_y.reshape(-1, 3, 3)
        self.width0 = float(p[..., 2].median())
        self.weight_floor = float(p[..., 0].min())
        return self

    def initial(self, v):
        n = v.shape[0]
        neg = torch.full((n, 1), float("-inf"))
        left = torch.cat([neg, v[:, :-1]], dim=1)
        right = torch.cat([v[:, 1:], neg], dim=1)
        is_peak = (v >= left) & (v >= right)
        score = torch.where(is_peak, v, v - 1e6)           # local maxima first (by height), other points only as fallback
        cols = torch.arange(v.shape[1])
        picks = []
        for _ in range(3):
            j = score.argmax(dim=1)
            picks.append(j)
            score = score.masked_fill((cols[None, :] - j[:, None]).abs() <= self.radius, float("-inf"))
        J = torch.stack(picks, dim=1)
        w = torch.gather(v, 1, J).clamp_min(self.weight_floor)
        c = X[J]
        s = torch.full_like(w, self.width0)
        return capstone_data.canonical_sort(torch.stack([w, c, s], dim=-1).reshape(n, 9))

    def predict(self, v):
        return capstone_data.mixture_ls_fit(v, self.initial(v), steps=self.refine_steps, lr=self.refine_lr,
                                            optimizer_wrapper=self.budget.wrap)


class AmortizedRegressor:
    # Alternative 2: MLP regressor on standardized samples, then the same refinement.
    def __init__(self, budget, train_steps=600, hidden=128):
        self.budget, self.train_steps, self.hidden = budget, train_steps, hidden

    def fit(self, train_x, train_y):
        SEAM.extend(capstone_data.split_of("mixture", row) for row in train_x)
        self.mu, self.sd = train_x.mean(dim=0), train_x.std(dim=0)
        torch.manual_seed(SEED)
        h = self.hidden
        self.net = nn.Sequential(nn.Linear(32, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 9))
        optimizer = self.budget.wrap(torch.optim.Adam(self.net.parameters(), lr=1e-3))
        z = (train_x - self.mu) / self.sd
        for _ in range(self.train_steps):
            optimizer.zero_grad(set_to_none=True)
            F.mse_loss(self.net(z), train_y).backward()
            optimizer.step()
        return self

    def initial(self, v):
        with torch.no_grad():
            return capstone_data.canonical_sort(self.net((v - self.mu) / self.sd))

    def predict(self, v):
        return capstone_data.mixture_ls_fit(v, self.initial(v), steps=REFINE_STEPS, lr=REFINE_LR,
                                            optimizer_wrapper=self.budget.wrap)


val_err = lambda pred: capstone_data.perm_invariant_error_per_example(pred, val_y)
approach = PeakInitLeastSquares(budget).fit(train_x, train_y)
val_pred = approach.predict(val_x)
VAL = {"peak init + 150 LS steps (chosen)": val_err(val_pred)}
VAL["peak init alone (no refinement)"] = val_err(approach.initial(val_x))
VAL["fixed init + 150 LS steps"] = val_err(capstone_data.mixture_ls_fit(
    val_x, capstone_data.mixture_baseline_init(val_x.shape[0]), steps=REFINE_STEPS, lr=REFINE_LR, optimizer_wrapper=budget.wrap))
amortized = AmortizedRegressor(budget).fit(train_x, train_y)
VAL["MLP regressor alone (600 steps)"] = val_err(amortized.initial(val_x))
VAL["MLP regressor + 150 LS steps"] = val_err(amortized.predict(val_x))
for name, e in VAL.items():
    print(f"validation error, {name}: mean {e.mean().item():.4f}, max {e.max().item():.4f}, #>0.02: {(e > 0.02).sum().item()}")
print(f"init width {approach.width0:.4f}, weight floor {approach.weight_floor:.4f}; steps used {budget.used}; elapsed {budget.elapsed():.2f}s")

**Locked test score (one call, after all fitting and selection).** The refinement inside `predict` is wrapped by the same budget.

In [ ]:
baseline = capstone_data.baseline_score("mixture")
assert capstone_data.test_call_count("mixture") == 0
test = capstone_data.final_test_score(approach.predict, n_boot=1000, alpha=0.05, task="mixture")
assert capstone_data.test_call_count("mixture") == 1
elapsed = budget.elapsed()
print(f"test mean permutation-invariant error {test.score:.4f}, 95% CI [{test.ci_low:.4f}, {test.ci_high:.4f}]; "
      f"baseline {baseline.score:.4f} [{baseline.ci_low:.4f}, {baseline.ci_high:.4f}]")
print(f"budget used {budget.used} of {MAX_STEPS}; elapsed {elapsed:.2f}s")

## Writeup

**Approach.** A structured initializer plus per-function least squares. For each function, the three highest local maxima of its 32 samples (with a 2-sample exclusion radius, and non-maxima only as a fallback) give the starting centers, the sampled heights the starting weights (floored at the smallest training weight), and the median training width the starting widths; then 150 batched Adam steps of `mixture_ls_fit` (weights, centers, log-widths; learning rate 0.02) fit each function to its samples, and the result is put in canonical (center-sorted) order. It works because the generator's separation rule makes every bump a distinct local maximum standing far above the 0.02 noise, so the initializer lands every component in its own basin, which is the one thing the fixed-initialization baseline cannot guarantee.

**Alternatives considered** (validation numbers printed above). (1) *The baseline's fixed initialization with the same 150 refinement steps*: its mean error is about three times larger ($0.0145$ versus $0.0043$) and 20 of the 200 functions end with errors above $0.02$ (worst about $0.22$: a component stuck between bumps or two components on one bump), so the initialization, not the optimizer, is the bottleneck. (2) *An amortized MLP regressor* (600 training steps on the 800 functions) *with the same refinement*: competitive but worse than the peak initializer on the mean ($0.0057$) with a few large failures, and it costs 600 training steps and most of the wall-clock time; alone (no refinement) it is far worse than the baseline, because 800 training functions cannot teach the network the exact per-function details. The peak initializer alone (no refinement) is also not enough, since its widths and weights are only rough.

**Evaluation.** Fitting used only `train_rows("mixture")` (the seam hashed every training row seen by either `fit` as `"train"`); every choice above was made on the 200 validation functions; the locked test functions were reached once, through `final_test_score`, after all choices. The approach's test error and 95% bootstrap interval are printed above against the committed fixed-initialization baseline's $0.0091$ (same rows, same metric), which it roughly halves, with the approach's whole interval below the baseline's point estimate. Total cost: 1,200 of the 2,000 budgeted steps (of which only 150 are needed by the chosen method at test time) and a few seconds.

**Limitations.** (i) The initializer relies on the generator's separation rule; if two bumps overlapped enough to share one local maximum, it would place a component on a noise peak or a shoulder, and refinement might not recover. I would test this with `simulate`-style functions generated with a smaller separation and track the error against the minimum center gap. (ii) The number of components is fixed at 3; a function with 2 or 4 bumps would be mis-specified, which I would test the same way. (iii) Larger noise would create spurious local maxima; the exclusion radius and the height ranking guard against small noise only, so I would sweep the noise level in simulation.

### Answer check

In [ ]:
assert test.score < baseline.score
assert budget.used <= MAX_STEPS
assert elapsed < MAX_SECONDS
assert len(SEAM) == 2 * 800 and not (set(SEAM) & {"val", "test"})
assert capstone_data.test_call_count("mixture") == 1
assert val_pred.shape == (200, 9) and bool((val_pred.reshape(-1, 3, 3)[..., 2] > 0).all())
means = {k: v.mean().item() for k, v in VAL.items()}
assert means["peak init + 150 LS steps (chosen)"] == min(means.values())         # chosen on validation